# 🌊 Week 4, Day 4 — June 11, 2026
## 10-Year Trajectory Analytics



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw", "processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata", "visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
master_v2 = pd.read_csv(DIRS['processed']+'/master_table_v2.csv')
df_s      = pd.read_csv(DIRS['processed']+'/species_clean_final.csv')
df_c      = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
print(f'master_v2: {master_v2.shape}')
print(f'species: {df_s.shape}')
print(f'climate: {df_c.shape}')

## Step 1: Build Year-by-Year Trend Series

In [ ]:
# Species count by year (from the bbox-filtered species dataset)
species_yearly = df_s.groupby('year')['species_common_name'].count().reset_index()
species_yearly.columns = ['year','Species_Count']
species_yearly = species_yearly.sort_values('year')

print('Species observations by year (2015–2026):')
print(species_yearly.to_string(index=False))
print()
print('NOTE: The rapid increase in 2023–2025 reflects greater citizen science')
print('reporting via iNaturalist (9,267 of 12,374 records). It does NOT necessarily')
print('mean species populations are growing — it means more were reported.')

In [ ]:
# Plastic trajectory
# Our dataset only covers 2015 but based on global trend (UNEP: ~8% annual growth)
# we simulate the projection to show the divergence
plastic_years = list(range(2015, 2027))
base_plastic_kg = 27621.35   # actual from bbox dataset
growth_rate = 0.08           # UNEP estimated 8% annual increase

plastic_trajectory = pd.DataFrame({
    'year': plastic_years,
    'Plastic_kg_estimated': [base_plastic_kg * (1 + growth_rate)**(y - 2015) for y in plastic_years]
})
plastic_trajectory['Source'] = plastic_trajectory['year'].apply(
    lambda y: 'Actual' if y == 2015 else 'Projected (UNEP +8%/yr)'
)
print('Plastic trajectory (actual 2015 + UNEP-projected 2016–2026):')
print(plastic_trajectory[['year','Plastic_kg_estimated','Source']].to_string(index=False))

In [ ]:
# SST trend from climate dataset
df_c['year'] = df_c['Date'].dt.year
sst_yearly = df_c.groupby('year')['SST (°C)'].mean().reset_index()
sst_yearly.columns = ['year','SST_C']

print('SST by year (2015–2023):')
print(sst_yearly.to_string(index=False))

In [ ]:
# Sea turtle focus — sensitive species indicator
turtles = df_s[df_s['species_common_name'].str.contains(
    'turtle|Turtle|Chelonia|chelonia|Caretta|Dermochelys|Eretmochelys|Lepidochelys',
    na=False, case=False
)]
turtle_yearly = turtles.groupby('year')['species_common_name'].count().reset_index()
turtle_yearly.columns = ['year','Turtle_Count']

print(f'Sea turtle records: {len(turtles)}')
print(turtle_yearly.to_string(index=False))

## Step 2: 4-Panel Trajectory Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.set_style('whitegrid')

# ── Panel 1: Species count trend ──
ax1 = axes[0][0]
ax1.plot(species_yearly['year'], species_yearly['Species_Count'],
         color='steelblue', linewidth=2.5, marker='o', markersize=6, label='All Species')
ax1.fill_between(species_yearly['year'], species_yearly['Species_Count'],
                 alpha=0.15, color='steelblue')
ax1.set_title('Marine Species Observations\n2015–2025 (Indian Ocean Bbox)', fontweight='bold')
ax1.set_xlabel('Year')
ax1.set_ylabel('Observation Count')
ax1.legend()
ax1.annotate('iNaturalist surge
(citizen science)', xy=(2023, 814),
             xytext=(2020, 1200), fontsize=8, color='gray',
             arrowprops=dict(arrowstyle='->', color='gray'))

# ── Panel 2: Plastic vs Species overlay (dual axis) ──
ax2 = axes[0][1]
ax2_r = ax2.twinx()

# Plastic (left axis)
actual = plastic_trajectory[plastic_trajectory['Source']=='Actual']
projected = plastic_trajectory[plastic_trajectory['Source']!='Actual']
ax2.bar(actual['year'], actual['Plastic_kg_estimated'],
        color='#e74c3c', alpha=0.9, width=0.4, label='Plastic (Actual)')
ax2.bar(projected['year'], projected['Plastic_kg_estimated'],
        color='#e74c3c', alpha=0.35, width=0.4, label='Plastic (Projected)')

# Species (right axis)
ax2_r.plot(species_yearly['year'], species_yearly['Species_Count'],
           color='steelblue', linewidth=2.5, marker='s', markersize=5, label='Species Count')
ax2.set_xlabel('Year')
ax2.set_ylabel('Plastic Weight (kg)', color='#e74c3c')
ax2_r.set_ylabel('Species Observations', color='steelblue')
ax2.set_title('Plastic Growth vs Species Observations\nDual-Axis Overlay 2015–2026', fontweight='bold')
ax2.legend(loc='upper left', fontsize=8)
ax2_r.legend(loc='upper right', fontsize=8)

# ── Panel 3: SST trend ──
ax3 = axes[1][0]
ax3.plot(sst_yearly['year'], sst_yearly['SST_C'],
         color='#e67e22', linewidth=2.5, marker='^', markersize=7)
ax3.fill_between(sst_yearly['year'], sst_yearly['SST_C'],
                 alpha=0.15, color='#e67e22')
z = np.polyfit(sst_yearly['year'], sst_yearly['SST_C'], 1)
p = np.poly1d(z)
ax3.plot(sst_yearly['year'], p(sst_yearly['year']),
         'r--', linewidth=1.5, alpha=0.7, label=f'Trend ({z[0]:+.4f}°C/yr)')
ax3.set_title('Sea Surface Temperature (SST)\n2015–2023 Indian Ocean', fontweight='bold')
ax3.set_xlabel('Year')
ax3.set_ylabel('SST (°C)')
ax3.legend()
ax3.set_ylim(27.5, 29.5)

# ── Panel 4: Sea Turtle population trend ──
ax4 = axes[1][1]
ax4.bar(turtle_yearly['year'], turtle_yearly['Turtle_Count'],
        color='#27ae60', alpha=0.8, edgecolor='darkgreen')
ax4.set_title('Sea Turtle Observations\n(Sensitive Species Indicator)', fontweight='bold')
ax4.set_xlabel('Year')
ax4.set_ylabel('Recorded Observations')
ax4.annotate('172 records\n(2025 spike)', xy=(2025, 172),
             xytext=(2022, 130), fontsize=8, color='darkgreen',
             arrowprops=dict(arrowstyle='->', color='darkgreen'))

plt.suptitle('June 11, 2026 — 10-Year Trajectory Analytics\nNorth Indian Ocean: Plastic × Species × SST',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week4_Day4_trajectory_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Trajectory plots saved ✅')

## Step 3: Identify Visual Divergence Points

In [ ]:
print('TRAJECTORY ANALYSIS — KEY FINDINGS')
print('='*55)
print()
print('1. PLASTIC TREND:')
print('   2015 actual: 27,621 kg in North Indian Ocean bbox')
print('   2026 projected: ~64,403 kg (+133% over 11 years)')
print('   Growth driver: UNEP global 8%/yr plastic leakage trend')
print()
print('2. SPECIES OBSERVATION TREND:')
print('   2015: 122 records → 2025: 4,757 records (39x increase)')
print('   ⚠️  This is a REPORTING surge, not a population surge.')
print('   iNaturalist onboarding (2020+) explains the jump post-2021.')
print('   Raw count alone cannot confirm species health.')
print()
print('3. SST TREND:')
sst_min = sst_yearly['SST_C'].min()
sst_max = sst_yearly['SST_C'].max()
sst_range = sst_max - sst_min
print(f'   Range: {sst_min:.2f}°C – {sst_max:.2f}°C (spread: {sst_range:.2f}°C)')
print('   No strong linear warming signal in 2015–2023 window.')
print('   Highest SST in 2020 (Marine Heatwave event).')
print()
print('4. DIVERGENCE POINT:')
print('   The decoupling of plastic growth (+133%) from reporting')
print('   growth (observational bias) is the central analytical')
print('   challenge. Week 5 Pearson + K-Means will disentangle this.')

In [ ]:
# Save trajectory data for Week 5 use
trajectory_df = species_yearly.merge(
    plastic_trajectory[['year','Plastic_kg_estimated']], on='year', how='outer'
).merge(
    sst_yearly, on='year', how='outer'
).merge(
    turtle_yearly, on='year', how='outer'
)
trajectory_df.to_csv(DIRS['processed']+'/trajectory_data.csv', index=False)
print(f'Trajectory dataset saved: trajectory_data.csv ({len(trajectory_df)} rows) ✅')